# Web search and usage metadata

## Goal

Compare a model-only answer with provider-hosted web search and inspect normalized token usage.

## Setup

Select the **Python (LangChain OpenAI)** kernel and run cells from top to bottom.
`OPENAI_API_KEY` and `OPENAI_MODEL` are read only from this course's root `.env`.
Shared settings are in `lesson_support.py`: up to 1024 output tokens, a 30-second
HTTP timeout, no HTTP retries, and LangSmith tracing disabled.
`make_model("another-api-model-id")` creates a different model with the same key.
The default text settings are intended for `gpt-5.6-luna`.
Creating a model does not call the API; `invoke` and consuming `stream` make paid requests.


## Without web search

In [ ]:
from pathlib import Path
import sys

# Find this course's root, whether Jupyter starts at the root or in notebooks/.
current_dir = Path.cwd().resolve()
course_root = next(
    (folder for folder in (current_dir, *current_dir.parents)
     if (folder / "course.json").is_file() and (folder / "lesson_support.py").is_file()),
    None,
)
if course_root is None:
    raise RuntimeError("Start Jupyter from the course root, as described in README.md.")
if str(course_root) not in sys.path:
    sys.path.insert(0, str(course_root))
from lesson_support import make_model

model = make_model()
print("Model:", model.model_name)


In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=model
)

In [ ]:
from langchain.messages import HumanMessage

question = HumanMessage(content="How up to date is your training knowledge?")

response = agent.invoke(
    {"messages": [question]}
)

In [ ]:
print(response['messages'][-1].content)

## Add web search

Use built-in OpenAI search through Responses API; a separate Tavily key is not needed.
OpenAI executes this provider tool. A Python @tool executes in your own application.
Search calls have a fee in addition to model tokens. Preserve clickable citations in displayed answers.

In [ ]:
# Same model and key, using Responses API for the built-in web_search tool.
search_model = make_model(use_responses_api=True)
web_search_tool = {"type": "web_search"}


In [ ]:
agent = create_agent(
    model=search_model,
    tools=[web_search_tool],
    system_prompt="Search the web for current facts and cite your sources.",
)

question = HumanMessage(content="What are the published dimensions of the Lada Niva Fora? Search the web and distinguish documented dimensions from estimates.")
response = agent.invoke({"messages": [question]})
print(response["messages"][-1].text)


In [ ]:
from pprint import pprint

pprint(response['messages'])

Inspect `response["messages"]` to see the reply and provider tool metadata.
Responses API and Chat Completions expose different raw metadata keys.
Prefer the final AIMessage's `usage_metadata` for normalized token counts.

In [ ]:
last_message = response["messages"][-1]

pprint(last_message.usage_metadata)